# Statistical significance checks on the pilot runs
50/100 questions, 20/50 template variations -> 1000/5000 total questions (20% of the benchmark)

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import logging

from gsm_benchmarker.results_analyser import MultiVariantMultiModelResultsAnalyser
from gsm_benchmarker.results_analyser.prompt_effect_analyser import PromptEffectAnalyser


logger = logging.getLogger('notebook')

plt.style.use('seaborn-v0_8-muted')
plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:
pp = Path("../../../../data/gsm-symbolic/outputs").resolve()
p_standard = pp / "mini_20x50x4__14_11/final"
p_sep = pp / 'mini_sep_new__20x50__20_12/final'
p_code = pp / 'mini_code_output_20x50__05_12/final'

In [ ]:
mres_standard = MultiVariantMultiModelResultsAnalyser(p_standard)
# mres_sep = MultiVariantMultiModelResultsAnalyser(p_sep)
mres_code = MultiVariantMultiModelResultsAnalyser(p_code)

In [ ]:
# pea_sep = PromptEffectAnalyser(mres_standard, mres_sep, "Anti-babbling prompt")
pea_code = PromptEffectAnalyser(mres_standard, mres_code, "Code output prompt")

## TO DO:
- both accuracies
- consider multiple comparisons correction
- analyse global effect
- new plots
- check on the full data

In [ ]:
_ = pea_code._baseline_mres.plot_question_difficulty_per_model()

In [ ]:
_ = pea_code._baseline_mres.plot_question_difficulty_histogram()

In [ ]:
glmm_results_df = pea_code._baseline_mres.analyse_variant_effect_glmm('main')
glmm_results_df

In [ ]:
from gsm_benchmarker.results_analyser.plotting_utils import plot_models_odds_ratios

_ = plot_models_odds_ratios(glmm_results_df, projected_alpha = 0.15)


In [ ]:
glmm_results_df_code = pea_code._experiment_mres.analyse_variant_effect_glmm('main')

# sort rows so that models appear in the same order as in baseline
model_order = glmm_results_df[['odds_ratio', 'model']].sort_values('odds_ratio', ascending=True).model
glmm_results_df_code = glmm_results_df_code.sort_values(
    by='model',
    key=lambda col: col.map({model: index for index, model in enumerate(model_order)})
).reset_index(drop=True)

_ = plot_models_odds_ratios(glmm_results_df_code, projected_alpha = 0.15, sort_models=False)